# 02 — Entraînement MobileNetV2 par Transfer Learning

Ce notebook **ne relance pas** l'entraînement (≈ 12 h 12 sur CPU) : il lit le
modèle sauvegardé et les artefacts produits par `training/train.py` et
`training/evaluate.py`, et explique chaque étape.

Pour réentraîner réellement :

```bash
python training/train.py
python training/evaluate.py
```


In [ ]:
import sys
from pathlib import Path

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))

import config
from utils.helpers import charger_json

configuration = charger_json(config.TRAINING_CONFIG_PATH, defaut={}) or {}
historique = charger_json(config.HISTORY_PATH, defaut={}) or {}
metriques = charger_json(config.METRICS_JSON_PATH, defaut={}) or {}
manifeste = charger_json(config.DATASET_MANIFEST_PATH, defaut={}) or {}

print("Configuration :", config.TRAINING_CONFIG_PATH.exists())
print("Historique    :", config.HISTORY_PATH.exists())
print("Métriques     :", (config.METRICS_JSON_PATH).exists())

## 1. Architecture

MobileNetV2 pré-entraînée sur ImageNet, sans sa couche de classification, puis
une tête adaptée à 38 classes. Le nombre de classes vient de `config.NUM_CLASSES`
(voir `training/train.py: construire_modele`).

In [ ]:
print(configuration.get("modele", "modèle non entraîné"))
print()
for cle, valeur in (configuration.get("architecture") or {}).items():
    print(f"{cle:<32} {valeur}")

In [ ]:
from tensorflow import keras

if config.MODEL_PATH.exists():
    modele = keras.models.load_model(config.MODEL_PATH)
    print("Sortie du modèle :", modele.output_shape, "→", modele.output_shape[-1], "probabilités")
    modele.summary(expand_nested=False)
else:
    print("models/best_model.keras absent : lancez training/train.py")

## 2. Hyperparamètres réellement utilisés

In [ ]:
phase1 = configuration.get("phase1", {})
phase2 = configuration.get("phase2_fine_tuning", {})
fine_tuning = phase2.get("fine_tuning", {})

lignes = {
    "Taille d'entrée": f"{config.IMAGE_SIZE[0]} × {config.IMAGE_SIZE[1]} × 3",
    "Batch size": configuration.get("batch_size"),
    "Optimiseur": configuration.get("optimiseur"),
    "Perte": configuration.get("perte"),
    "Phase 1 — époques": f"{phase1.get('epochs_effectues')} (max {phase1.get('epochs_max')})",
    "Phase 1 — learning rate": phase1.get("learning_rate"),
    "Phase 2 — époques": phase2.get("epochs"),
    "Phase 2 — learning rate": phase2.get("learning_rate"),
    "Couches examinées (fine-tuning)": fine_tuning.get("couches_examinees"),
    "Couches dégelées (drapeau)": fine_tuning.get("couches_degelees_par_drapeau"),
    "Couches réellement réentraînées": fine_tuning.get("couches_entrainables_avec_poids"),
    "BatchNormalization gelées": f"{fine_tuning.get('batch_normalization_gelees')} / {fine_tuning.get('batch_normalization_total')}",
    "Graine": configuration.get("graine"),
    "Durées (min)": f"{phase1.get('duree_min')} + {phase2.get('duree_min')}",
}
for cle, valeur in lignes.items():
    print(f"{cle:<34} {valeur}")

## 3. Deux phases

| Phase | Base MobileNetV2 | Ce qui apprend | Learning rate |
| --- | --- | --- | --- |
| 1 — extraction de caractéristiques | entièrement gelée | la tête (GAP → Dense 128 → Dense 38) | 1e-3 |
| 2 — fine-tuning | 30 dernières couches examinées, BatchNormalization gelées | 19 couches dégelées, dont 10 porteuses de poids | 1e-5 |

La phase 2 repart du **meilleur modèle de la phase 1** et ne l'écrase que si
l'accuracy de validation s'améliore (ModelCheckpoint avec seuil initial).

## 4. Courbes d'apprentissage

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(config.TRAINING_CURVES_PATH)))

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(historique.get("loss", [])) + 1)
figure, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(epochs, historique.get("accuracy", []), label="train")
axes[0].plot(epochs, historique.get("val_accuracy", []), label="validation")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("Époque"); axes[0].legend()
axes[1].plot(epochs, historique.get("loss", []), label="train")
axes[1].plot(epochs, historique.get("val_loss", []), label="validation")
axes[1].set_title("Perte"); axes[1].set_xlabel("Époque"); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"Époques : {len(historique.get('loss', []))}")
print(f"Meilleure accuracy de validation : {max(historique.get('val_accuracy', [0])):.4f} "
      f"(époque {int(max(range(len(historique['val_accuracy'])), key=historique['val_accuracy'].__getitem__)) + 1})")

## 5. Résultats sur l'ensemble de test

In [ ]:
global_ = metriques.get("global", {})
for cle, valeur in global_.items():
    print(f"{cle:<22} {valeur}")

par_classe = metriques.get("par_classe", {})
if par_classe:
    tri = sorted(par_classe.items(), key=lambda item: item[1]["f1_score"])
    print("\nClasses les plus difficiles :")
    for classe, valeurs in tri[:3]:
        print(f"  {classe:<50} F1={valeurs['f1_score']:.4f} (support {valeurs['support']})")
    print("Classes les mieux reconnues :")
    for classe, valeurs in list(reversed(tri))[:3]:
        print(f"  {classe:<50} F1={valeurs['f1_score']:.4f} (support {valeurs['support']})")

In [ ]:
display(Image(filename=str(config.CONFUSION_MATRIX_PATH)))
display(Image(filename=str(config.FIGURES_DIR / "per_class_performance.png")))

## 6. Prédire une image

In [ ]:
from utils.helpers import nom_affichage
from utils.predictor import obtenir_predictor

predictor = obtenir_predictor()
predictor.charger()
print("Modèle chargé :", predictor.version, "|", len(predictor.classes), "classes")

exemple = sorted((config.PROCESSED_DATA_DIR / "test" / "Tomato___Late_blight").iterdir())[0]
resultat = predictor.predire(exemple)
print(f"Image          : {exemple.name}")
print(f"Classe prédite : {nom_affichage(resultat['classe'])} ({resultat['confiance'] * 100:.1f} %)")
print(f"Durée          : {resultat['duree_ms']} ms")
for element in resultat["classement"][:5]:
    print(f"  {nom_affichage(element['classe']):<34} {element['probabilite'] * 100:5.1f} %")

## 7. Conclusion

- architecture volontairement légère (2,4 M de paramètres) ;
- deux phases : tête seule, puis fine-tuning des dernières couches avec un
  learning rate 100 fois plus faible ;
- résultats obtenus sur 8 153 images de test **jamais vues** pendant
  l'entraînement, entièrement recalculables avec `training/evaluate.py`.
